# Session 3 — Solutions

Worked answers with the reasoning spelled out. The code is the small part; each solution ends by
naming where the idea returns later in the course.

Outputs here are from one real run. Yours will differ in wording — and in exercises 3 and 5, in the
numbers. Every conclusion below is about behaviour that holds across runs, and where a number is
inside the noise, the text says so rather than pretending otherwise.

In [1]:
import json
import re
import subprocess
import sys
from pathlib import Path
from typing import Literal, Optional

import yaml
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError

HERE = Path.cwd()
sys.path.insert(0, str(HERE / "project"))
from config import settings                      # noqa: E402
from schemas import TicketCategory, TriageResult  # noqa: E402

client = genai.Client()
TICKETS = yaml.safe_load((settings.DATA_DIR / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in TICKETS}

TRIAGE_PROMPT = yaml.safe_load(
    (HERE / "project" / "prompts.yml").read_text(encoding="utf-8"))["triage_prompt"]


def block(ticket: dict) -> str:
    return f"<ticket>\nSubject: {ticket['subject']}\n{ticket['body']}\n</ticket>"


def call(schema, ticket):
    """Returns (parsed_or_None, usage). Never hides a None — see exercise 5."""
    r = client.models.generate_content(
        model=settings.MODEL,
        contents=block(ticket),
        config=types.GenerateContentConfig(
            system_instruction=TRIAGE_PROMPT,
            response_mime_type="application/json",
            response_schema=schema))
    return r.parsed, r.usage_metadata


def ask(schema, ticket):
    return call(schema, ticket)[0]


print(f"{len(TICKETS)} tickets · model {settings.MODEL}")

20 tickets · model gemini-3.1-flash-lite


## ⭐ Exercise 1 — The confidence field

In [2]:
class TriageWithConfidence(TriageResult):
    confidence: Literal["low", "medium", "high"] = Field(
        description=("How sure you are of the category. Answer low when the ticket raises "
                     "several separate issues, or when two categories fit equally well."))


for tid in ("TCK-003", "TCK-001"):
    r = ask(TriageWithConfidence, by_id[tid])
    print(f"{tid}: category={r.category:14} confidence={r.confidence:8} — {r.summary}")

TCK-003: category=order_status   confidence=medium   — Customer demands immediate shipment or refund for delayed order ORD-5002.


TCK-001: category=order_status   confidence=high     — Customer inquiring about the delivery status of order ORD-5001.


### The answer — and it is not the one the exercise sets you up for

`TCK-003` is the ticket the whole session argues about: `complaint` and `order_status` both genuinely
fit, and two reasonable humans disagree. **The model usually reports `high` confidence on it anyway.**

That is the finding. **Self-reported confidence is not calibrated.** It reflects how fluent the answer
felt to produce, not how likely it is to be right — and a model is most confidently wrong exactly
where the ambiguity is invisible to it, because nothing in its training says ShopWise has a house rule
about whose fault it was.

Writing a good `description` genuinely helps: without one you get "high" on literally everything,
because the model has no idea what your bar is. But it raises the floor; it does not make the number
trustworthy. Section 6.3 of the notebook is the same shape of lesson — the schema will happily give
you a field that *looks* like a signal.

**A field is only worth having if code acts on it.** Name the action or delete the field:

| `confidence == "low"` could mean | is that real? |
|---|---|
| route to a human queue | yes — cheap, and the cost of being wrong is small |
| ask the customer a clarifying question | yes, if your product has anywhere to put one |
| retry with the stronger model | yes, and now you can price it: `MODEL_STRONG` costs 3× |
| show a warning icon in a dashboard | only if a human acts on the icon |
| nothing | then delete the field — you are paying output tokens for decoration |

Session 10 uses this pattern for real: `confidence` becomes a routing condition on a graph edge, and
the "ask a clarifying question" branch becomes a node.

## ⭐⭐ Exercise 2 — The angry detector

In [3]:
SYSTEM = """You are the customer support assistant for ShopWise, an online electronics store.
Reply warmly and plainly. Never promise a refund, delivery date or any outcome —
say a human colleague will follow up."""

APOLOGY = ("I am sorry — you should not have had to write in again about this, "
           "and I understand why you are frustrated.\n\n")


def draft_reply(ticket, triage) -> str:
    """Draft the reply. The apology is a code decision, not a prompt request."""
    body = client.models.generate_content(
        model=settings.MODEL,
        contents=block(ticket),
        config=types.GenerateContentConfig(system_instruction=SYSTEM)).text

    if triage.sentiment == "angry":          # <- the entire point of the session
        return APOLOGY + body.strip()
    return body.strip()


for tid in ("TCK-003", "TCK-001"):
    t = by_id[tid]
    tri = ask(TriageResult, t)
    reply = draft_reply(t, tri)
    print(f"=== {tid} (sentiment={tri.sentiment}) ===")
    print(reply[:300], "\n")

# And the test you can now write, which is the argument in one line:
angry = ask(TriageResult, by_id["TCK-003"])
if angry.sentiment == "angry":
    assert draft_reply(by_id["TCK-003"], angry).startswith(APOLOGY)
    print("assert passed — the apology is guaranteed, not likely")

=== TCK-003 (sentiment=angry) ===
I am sorry — you should not have had to write in again about this, and I understand why you are frustrated.

Hello,

I am so sorry to hear that you have been waiting for a response and that your order is still showing as processing. I completely understand your frustration, especially after reaching 



=== TCK-001 (sentiment=calm) ===
Hi there! Thanks for reaching out to us. 

I’m sorry to hear that you haven’t seen any updates on your tracking for your AeroSound Pro headphones yet. I understand how frustrating it is to wait for an order without knowing where it is.

I have gone ahead and flagged your order (ORD-5001) for review. 



assert passed — the apology is guaranteed, not likely


### Why the `if` wins

| | in the prompt | in the code |
|---|---|---|
| happens every time | no — it is a tendency | **yes** |
| you can test it | not really | `assert reply.startswith(APOLOGY)` |
| a non-engineer can change the wording | yes, in `prompts.yml` | needs a deploy |
| shows up in a diff | yes | yes |
| survives a model swap | unknown until you re-test | **yes** |

The prompt version is not useless — it produces *better-tailored* apologies, because the model writes
one that fits the specific complaint. The code version produces the *same* apology every time, which
is duller and **guaranteed**.

**The rule this is a special case of: anything a customer must never see, or must always see, belongs
in code.** Prompts are for shaping; code is for guaranteeing. You have already met the strongest form
of this in section 4.5 — `schemas.py` forces `needs_human` to True for money categories in a
`model_validator`, where no prompt and no jailbreak can reach it. That distinction is the spine of
session 7, where the things that must never happen stop being requests and become middleware.

Note also what made this possible at all: `sentiment` is a `Literal`, so `== "angry"` is safe. With
`sentiment: str` you would be back to matching on whatever word the model chose.

## ⭐⭐⭐ Exercise 3 — Build an objective metric

The whole point is that the ground truth costs nothing: a regular expression over the ticket body.
No labelling meeting, no convention to agree, nothing to argue about in review.

In [4]:
class TriageWithOrders(TriageResult):
    order_ids: list[str] = Field(
        default_factory=list,
        description=("Every order ID mentioned, in ORD-XXXX format, exactly as written. "
                     "Return an empty list if the ticket mentions none. "
                     "Never invent one that is not in the text."))


def truth(ticket) -> list[str]:
    return re.findall(r"ORD-\d+", ticket["body"])


exact = missed = hallucinated = invalid = 0
wrong = []

for t in TICKETS:
    want = truth(t)
    got = ask(TriageWithOrders, t)
    if got is None:                                   # section 6.3 — never assume
        invalid += 1
        wrong.append((t["id"], want, "NO RESPONSE"))
        continue
    if got.order_ids == want:
        exact += 1
    else:
        wrong.append((t["id"], want, got.order_ids))
    # These two are NOT the same failure and must not be averaged together.
    missed += len(set(want) - set(got.order_ids))
    hallucinated += len(set(got.order_ids) - set(want))

print(f"exact match : {exact}/{len(TICKETS)} = {exact / len(TICKETS) * 100:.0f}%")
print(f"missed IDs  : {missed}     (in the text, not extracted)")
print(f"invented IDs: {hallucinated}     (extracted, not in the text)")
print(f"unusable    : {invalid}")
for tid, want, got in wrong:
    print(f"  {tid}: wanted {want} got {got}")

exact match : 20/20 = 100%
missed IDs  : 0     (in the text, not extracted)
invented IDs: 0     (extracted, not in the text)
unusable    : 0


### What the number is worth

Extraction accuracy is usually much higher than category accuracy, and the reason is the whole lesson:
**there is nothing to disagree about.** `ORD-5002` is in the text or it is not. Nobody has a house rule
about it. When this metric drops, the model got worse — full stop, no meeting required.

**Missed and invented are not the same failure and must never be averaged into one score.**

| | what happens downstream (session 5) | severity |
|---|---|---|
| **missed** an ID | the tool is not called; the ticket goes to a human | annoying |
| **invented** an ID | the tool is called with a fake ID, and returns *something* | dangerous |

A single "accuracy" figure hides that difference completely, which is how a team ships the version
that hallucinates less often but more expensively. Report them separately, always.

### Empty list, not `None` — and it is not a close call

```python
for oid in triage.order_ids:        # empty list: just works, does nothing
    lookup(oid)

if triage.order_ids is not None:    # None: every consumer, forever, must remember this
    for oid in triage.order_ids:
        lookup(oid)
```

`None` pushes a check into every place the field is used, and someone will eventually forget it and
ship a `TypeError`. An empty list is *the same thing the loop does anyway* — nothing.

Two details that make this work in practice:

- **`default_factory=list`**, not `= []`. A mutable default is shared between instances — a classic
  Python bug. Pydantic stops you; know why.
- **"exactly as written" and "never invent one"** in the description. Both earn their place: without
  the first you get `ORD5001` or `ord-5001`; without the second you get a plausible ID on a ticket
  that never mentioned one. And remember section 6.3 — a `pattern=` regex in the schema would *not*
  have been enforced by the decoder, so the description is doing the real work here.

### Which number goes in front of the manager

Both, in this order: *"extraction is 20/20 — every order ID in every ticket, none invented. Category is
18/20, and two of those misses are label disagreements we could settle by changing our own convention
rather than the prompt."* The objective number establishes that the system works; the conventional one
is then a conversation about definitions rather than about competence. Leading with the conventional
number alone invites a debate you cannot win with engineering.

And note how much more *useful* the objective number is here, precisely because it is boring. 20/20
with zero inventions is a fact you can act on: session 5 can look these IDs up. "90% on category" is a
number you have to explain.

## ⭐⭐ Exercise 4 — Earn your few-shot

In [5]:
done = subprocess.run([sys.executable, "score_triage.py", "--diff"],
                      cwd=HERE, capture_output=True, text=True, timeout=1800)
print(done.stdout)


WITHOUT examples: 18/20 = 90%
  0 unusable response(s) · 1135 output tokens · $0.0028 for the run ($0.14 per 1,000 tickets)

  ticket    hand label         assistant said
  TCK-003   complaint          order_status
  TCK-005   return             warranty

  commonest confusions: complaint -> order_status (x1), return -> warranty (x1)

WITH examples: 17/20 = 85%
  0 unusable response(s) · 1129 output tokens · $0.0038 for the run ($0.19 per 1,000 tickets)

  ticket    hand label         assistant said
  TCK-005   return             warranty
  TCK-012   shipping           order_status
  TCK-015   warranty           product_question

  commonest confusions: return -> warranty (x1), shipping -> order_status (x1), warranty -> product_question (x1)

The examples bought -1 ticket(s) = -5 percentage points, for +33% cost.
  fixed : TCK-003
  BROKE : TCK-012, TCK-015  <- examples can cost you tickets too



### Part A — model wrong, or label arguable?

The persistent misses are worth reading carefully, because they are different kinds of miss.

**`TCK-003` — label arguable.** A furious all-caps message about an unshipped order. Hand-labelled
`complaint`, model says `order_status`. Both are defensible; ShopWise's convention is that when the
company's conduct is the subject, that is the complaint. **A convention to transmit, not an error to
fix** — a *conventional* metric miss in section 8.1's table.

**`TCK-005` — genuinely hard, and also not a language problem.** Labelled `return`, model says
`warranty`. A faulty item the customer wants to send back is *both*, and which one it is depends on
whether ShopWise processes faults under warranty or under returns. That is a business-process fact.
No amount of prompting lets a model guess your internal process: **write it in the prompt or accept
the miss.**

### Parts B and C — what the examples actually bought

Whatever your three examples do, run the comparison twice before you believe it. Appendix G measured
this exact set moving between 15 and 18 across identical runs — so a one-ticket "improvement" is
indistinguishable from nothing, and reporting it as a win is how teams talk themselves into permanent
token costs.

The authored answer: **the three shipped examples move the total by zero.** They fix one ticket and
break another. So they ship off, and the number is written next to the flag in `config.py` so nobody
silently re-runs the experiment in six months.

**The general principle, worth more than the specific result:**

1. **Write the rule before you show the example.** A rule costs tokens once and a reader can check it;
   the rules in `triage_prompt` are what moved 15/20 to 18/20.
2. **Examples break things they were not aimed at.** You only see it by scoring the whole set.
3. **A null result is a result** — and a cheap one, because it stops you paying forever for nothing.

## ⭐⭐ Exercise 5 — The constraint that usually works

Ten tickets rather than twenty, to keep the solution quick; the rate is what matters, not the count.

In [6]:
from pydantic import create_model

SAMPLE = TICKETS[:10]
PRESSURE = "\n\nBe thorough and detailed in the summary."


def capped(maxlen: int, desc: str | None = None):
    """Build a TightSummary variant at a given cap, optionally with a description."""
    field = Field(max_length=maxlen, **({"description": desc} if desc else {}))
    return create_model("Capped", category=(TicketCategory, ...), summary=(str, field))


def measure(schema, tickets, pressure: str = ""):
    """Returns (invalid_rate, summary_lengths, output_tokens)."""
    bad, lengths, tokens = 0, [], 0
    for t in tickets:
        r = client.models.generate_content(
            model=settings.MODEL, contents=block(t) + pressure,
            config=types.GenerateContentConfig(
                system_instruction=TRIAGE_PROMPT,
                response_mime_type="application/json", response_schema=schema))
        tokens += r.usage_metadata.candidates_token_count
        # .text is there even when .parsed is None — that is how you see the truth.
        try:
            lengths.append(len(json.loads(r.text)["summary"]))
        except Exception:
            pass
        bad += r.parsed is None
    return bad / len(tickets), lengths, tokens


# 1 + 2 — does the model even see the constraint?
print("STEP 1 & 2 — ordinary prompt, no pressure")
for cap in (40, 50, 60):
    rate, lengths, _ = measure(capped(cap), SAMPLE)
    print(f"  max_length={cap}: {rate * 100:3.0f}% unusable · lengths {sorted(lengths)}")

STEP 1 & 2 — ordinary prompt, no pressure


  max_length=40:   0% unusable · lengths [24, 31, 31, 34, 34, 35, 37, 38, 38, 40]


  max_length=50:   0% unusable · lengths [40, 40, 41, 42, 42, 44, 48, 48, 48, 49]


  max_length=60:   0% unusable · lengths [41, 41, 42, 47, 51, 53, 53, 54, 56, 57]


### Steps 1 and 2 — the constraint appears to work

Zero unusable, at every cap. And look at the lengths: they **track the cap**. At 40 you get summaries
in the thirties; at 60 you get summaries in the forties and fifties. Nothing violated its limit.

So the model can plainly see `maxLength` — it is in the JSON Schema, it is sent, it is read. **It is
advice the model takes.** What it is not is a gate.

A test suite built on these twenty tickets would pass, forever, and tell you nothing.

In [7]:
# 3 — now push on it, exactly as a real ticket eventually will
print("STEP 3 — one extra sentence of pressure")
rate, lengths, _ = measure(capped(60), SAMPLE, PRESSURE)
print(f"  max_length=60 + pressure: {rate * 100:3.0f}% unusable · lengths {sorted(lengths)}")

STEP 3 — one extra sentence of pressure


  max_length=60 + pressure:  10% unusable · lengths [42, 50, 51, 53, 56, 57, 59, 60, 60, 63]


### Step 3 — and there it is

One sentence. Not an attack, not a jailbreak — the kind of sentence a colleague adds to a prompt on a
Tuesday because replies were feeling thin. The rate goes from zero to something, and every one of
those is a `.parsed` of `None` with no exception and no log line.

**This is why the failure reaches production.** It is not that the constraint never works. It is that
it works exactly until the day it does not, and nothing tells you when that day arrives.

In [8]:
# 4 — three fixes, measured, with cost
print("STEP 4 — three fixes, all under pressure")
SAID_SO = "At most 60 characters. Count them. Truncate rather than run over."
for label, schema in (("constraint only        ", capped(60)),
                      ("+ said so in description", capped(60, SAID_SO)),
                      ("relaxed to 200 chars   ", capped(200, "One short sentence for a dashboard row."))):
    rate, lengths, tokens = measure(schema, SAMPLE, PRESSURE)
    print(f"  {label}: {rate * 100:3.0f}% unusable · {tokens} output tokens · "
          f"longest {max(lengths) if lengths else 0}")

STEP 4 — three fixes, all under pressure


  constraint only        :  10% unusable · 312 output tokens · longest 63


  + said so in description:   0% unusable · 273 output tokens · longest 60


  relaxed to 200 chars   :   0% unusable · 399 output tokens · longest 181


### The decision

The three fixes are not equivalent, and the difference is the entire point:

| fix | what it changes | honest verdict |
|---|---|---|
| say the limit in the `description` | the model sees the rule twice instead of once | makes it **rarer**. It is still advice |
| relax the constraint | the rule now matches what the model actually does | makes it **impossible** — there is nothing left to violate |
| repair loop (section 9.3) | asks again with the error attached | works, and multiplies your bill exactly when things are already going badly |

Ship the relaxed constraint, and put the real requirement in the description. Keep the repair loop for
rules you genuinely cannot relax — a downstream system's hard limit, a legal cap on a field length.

Look closely at the description run before you move on: it came back at **0% unusable, longest summary
exactly 60**. The model counted, and got it right — this time, on these ten tickets. That is a very
persuasive-looking result and it is still not a guarantee. If you would not bet the incident channel on
it, it is not a control.

**And notice what the middle row is really telling you.** Repeating a rule to a model reduces the
failure rate. It does not change the *kind* of thing the rule is. Anything you cannot afford to have
fail must stop being a request and become something the code enforces — which is the `model_validator`
from section 4.5, and the whole of session 7.

> **The general shape:** an invariant is only worth writing if something enforces it. Ask of every
> constraint you add — *who checks this, and what happens when it fails?* If the answer is "the model
> usually remembers", you have written a comment that throws away data.

### Where the None-check belongs

`project/utils.py` already answers this: `checked()`, called on every response before anything else
touches it. One place, so no future call site can forget — and it raises `ModelReturnedNothing` with
the `finish_reason` in the message, so the postmortem takes a minute instead of an afternoon.

That is the pattern to copy: **turn a silent `None` into a loud, named exception at the boundary.**

## 💼 Interview questions — answer sketches

Written the way you would say them.

---

**1. When do you demand structured output, and when is prose the product?**

> "Structure when code consumes it — branching, storing, counting, or feeding the next call. Prose when
> a human is the consumer and the writing *is* the deliverable.
>
> Our support assistant does both in one pipeline. Triage is structured, because we route on it and put
> it in a dashboard. The customer's reply is free text, because a schema would only get in the way of
> writing a good email."

---

**2. What is the difference between asking a model for JSON and passing it a schema?**

> "Asking is a request; a schema is a mechanism. With `response_schema` the provider constrains
> decoding — at each token it masks anything that could not continue a valid document, so a value
> outside my enum is unreachable rather than unlikely. Asking in the prompt gets you well-formed JSON
> with invented values, which is worse than a parse error because it fails quietly downstream.
>
> The caveat I'd add: constrained decoding enforces *structure*, not *content*. `max_length` and
> `pattern` are not enforced by the decoder — Pydantic rejects the answer afterwards and the SDK hands
> you `None`."

---

**3. The model returns a category that is not in your list. Name the two places you could catch it,
and which you prefer.**

> "In the schema, or in my code after the fact. I strongly prefer the schema: with a `Literal`, the
> constrained decoding means an invalid value cannot be produced, so there is no error path to write
> and no branch to test. Catching it afterwards means it already happened, and now I have to decide
> what to do with a bad answer I paid for.
>
> The schema does not fix a *wrong* value from a valid list, though. That is what the eval set is
> for."

---

**4. Your schema says `max_length=60` and you are getting no results at all, with no errors. What
happened?**

> "The provider ignored the length constraint — it only enforces structure — the model wrote a longer
> string, Pydantic refused it on the way in, and the SDK set `.parsed` to `None` rather than raising.
> So every call 'succeeds' and nothing is usable.
>
> Diagnose it by validating the raw text yourself to see the real `ValidationError`. Fix it by putting
> the limit in the field description, relaxing the constraint, or adding a repair loop. And check
> `.parsed` for None at one chokepoint so it can never be silent again."

---

**5. You add three few-shot examples and accuracy is unchanged. Name two things that could be wrong.**

> "First: the rule was already in the prompt, so the examples taught nothing new — examples transmit a
> convention the instructions failed to state, and if the instructions already state it you are paying
> for a restatement.
>
> Second: the examples do not resemble the failures. Examples fix judgement on *borderline* cases; if
> mine are obvious tickets they change nothing.
>
> And a third worth saying out loud: my eval set may be too small to see the effect. Twenty items move
> by a couple between identical runs, so 'unchanged' might mean 'smaller than my noise'."

---

**6. Where would you put a rule that says "a refund ticket always goes to a human" — the prompt, the
schema, or the code? Why?**

> "The code — specifically a `model_validator` on the schema, so it runs at the boundary where the
> model's answer becomes an object, before any call site sees it.
>
> The prompt is a request: it holds most of the time, and 'most of the time' is not a control on
> something that moves money. The schema's *types* cannot express it either, because it is a rule
> across two fields. A validator is four lines, it runs on every path including the ones added next
> year, and it survives a model swap. Models advise; code decides."

---

## The one from the answer key

**Which of v0.2's two calls should stay free-text?**

`draft_reply` — the prose **is** the product. It is the email a customer reads, and its quality is
measured by whether a human feels well treated, which no schema captures.

`triage` gets a schema because nothing downstream can consume a paragraph: you cannot route on it,
count it, or put it in a dashboard row.

**The general test:** ask *who reads this output.* Code reads the triage, so triage gets a form. A
person reads the reply, so the reply stays prose. Getting this backwards is how you end up with a
support bot that emails customers JSON — which is funnier to describe than to explain to your
manager.